# 19 A small model that loops while it thinks: repair it with a decoding setting or with a LoRA? (Google Colab, GPU)

A 0.8B-parameter model that thinks before it answers can fall into a loop and never finish. In the weight-editing case the book's research notes cover, a frontier model edited 0.016% of such a model's weights and lifted its score on MATH-500 from 7.9% to 48.1%; the same model with thinking switched off scored 45.8%, and the model card's recommended sampling settings alone recovered most of the gap in an independent re-run. Nobody had compared the edit with the ordinary repairs. This notebook runs that comparison on a **public analog** that fits a free Colab GPU: `Qwen/Qwen3-0.6B`, which also loops in thinking mode (its model card says greedy decoding can lead to "endless repetitions").

**What it measures, on the same MATH-500 problems for every arm.** Accuracy, the **cut rate** (the share of answers that hit the token limit without finishing) and the **loop rate** (the share whose last 1,000 tokens contain the same 20-token sequence three times or more). Part 1 compares four settings of the unchanged model: thinking on with greedy decoding (the card says not to), thinking on with the card's sampling settings, greedy decoding with a repetition penalty, and thinking off. Part 2 trains a rank-8 **LoRA** (a small trainable add-on to the attention weights) on the model's own finished, correct reasoning traces from GSM8K, and repeats three of the arms with it.

**What it does not do.** It does not reproduce the published edit (that targets a different model, Qwen3.5-0.8B) and it does not run the editing procedure. A difference-of-means "loop direction" edit is the third repair the notes propose; it is not included because choosing its layers needs a run of its own. Read the result as: how much of a loop problem does each ordinary repair remove on this model, at this size and token limit.

**Runtime.** Colab: Runtime, Change runtime type, T4 GPU. Roughly one hour with the defaults (100 evaluation problems, 2,048 new tokens). That figure is an estimate from the speed of a small model on a T4, not a measurement: the first printed batch tells you the real rate, and `N_EVAL` and `MAX_NEW` are the knobs. A smaller `N_EVAL` widens every interval.

In [ ]:
SMOKE = False   # True: quick check that the code runs (4 problems, 48 new tokens). Leave False for the real run.
import os
SMOKE = SMOKE or os.environ.get("SMOKE") == "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less memory fragmentation; must be set before torch starts
if not SMOKE:
    !pip -q install transformers==4.57.6 peft==0.17.1 accelerate 2>&1 | tail -n 1
import json, math, re, time, random, collections, gc
import numpy as np, pandas as pd, torch
from scipy.stats import binomtest
from huggingface_hub import hf_hub_download
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if dev == "cuda" else "none (CPU: only the smoke test is feasible)")
import transformers, sys
_gpu = torch.cuda.is_available()
print("preflight | python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPU:", (torch.cuda.get_device_name(0) + f", {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB") if _gpu else "none", flush=True)
if not SMOKE and not _gpu:
    raise SystemExit("No GPU. In Colab: Runtime, Change runtime type, choose a GPU (T4 is enough), then run all again. CPU and TPU runtimes are not supported.")

N_EVAL   = 4 if SMOKE else 100     # MATH-500 problems per arm
N_TRACE  = 6 if SMOKE else 300     # GSM8K training problems used to collect traces for the LoRA
MAX_NEW  = 48 if SMOKE else 2048   # new tokens per answer; the cut rate depends on this number
BATCH    = 4 if SMOKE else 32
SEED     = 0

**Goal.** Load the problems. MATH-500 (`HuggingFaceH4/MATH-500`) is the evaluation set; GSM8K training problems (`openai/gsm8k`) supply the traces the LoRA learns from, so the LoRA never sees an evaluation problem. The sample of evaluation problems is drawn with a fixed seed.

In [ ]:
path = hf_hub_download("HuggingFaceH4/MATH-500", "test.jsonl", repo_type="dataset")
math500 = [json.loads(l) for l in open(path)]
rng = random.Random(SEED); ev = rng.sample(math500, N_EVAL)
gp = hf_hub_download("openai/gsm8k", "main/train-00000-of-00001.parquet", repo_type="dataset")
gsm = pd.read_parquet(gp).sample(N_TRACE, random_state=SEED)
gsm_items = [dict(problem=q, answer=a.split("####")[-1].strip().replace(",", "")) for q, a in zip(gsm["question"], gsm["answer"])]
eval_items = [dict(problem=r["problem"], answer=r["answer"]) for r in ev]
print(len(eval_items), "MATH-500 problems,", len(gsm_items), "GSM8K problems; by level:", dict(collections.Counter(r["level"] for r in ev)))

**Goal.** Define the three things measured. A **correct** answer is one whose last `\\boxed{...}` (after the closing `</think>` when the model was thinking) matches the reference after light normalization (spaces, `\\dfrac`, units text, trailing zeros). This grader is crude: it will mark some equivalent answers wrong, so absolute accuracies are lower than a careful grader would give; it treats every arm the same, so the comparisons stay fair. An answer that never closes `</think>` has no answer region and counts as wrong. The **loop flag** looks for the same 20-token sequence at least three times in the last 1,000 generated tokens; it is a heuristic, applied to finished and unfinished answers alike.

In [ ]:
def last_boxed(s):
    i = s.rfind("\\boxed{")
    if i < 0: return None
    j, depth = i + len("\\boxed{"), 1; out = []
    while j < len(s) and depth:
        ch = s[j]; depth += (ch == "{") - (ch == "}")
        if depth: out.append(ch)
        j += 1
    return "".join(out) if depth == 0 else None
def norm(a):
    a = re.sub(r"\\text\{([^}]*)\}", r"\1", a.strip())
    for x, y in (("\\dfrac", "\\frac"), ("\\tfrac", "\\frac"), ("\\left", ""), ("\\right", ""), ("\\!", ""), ("^\\circ", ""), ("^{\\circ}", ""), ("\\$", ""), ("$", ""), (" ", ""), ("\\%", ""), ("%", "")):
        a = a.replace(x, y)
    return a.rstrip(".")
def same(a, b):
    a, b = norm(a), norm(b)
    if a == b: return True
    try: return abs(float(a.replace(",", "")) - float(b.replace(",", ""))) < 1e-6
    except ValueError: return False
def is_correct(text, ref, thinking):
    if thinking: region = text.split("</think>")[-1] if "</think>" in text else ""
    else: region = text
    ans = last_boxed(region)
    return ans is not None and same(ans, ref)
def looped(ids, n=20, window=1000, times=3):
    ids = ids[-window:]; seen = collections.Counter(tuple(ids[i:i + n]) for i in range(max(0, len(ids) - n + 1)))
    return bool(seen) and max(seen.values()) >= times
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h
# tiny self-test of the grader
assert last_boxed("so \\boxed{\\frac{1}{2}} done") == "\\frac{1}{2}" and same("\\dfrac{1}{2}", "\\frac{1}{2}") and same("5.0", "5")
assert looped(list(range(50)) + [7, 8, 9] * 20 + list(range(100, 130)) * 4) and not looped(list(range(500)))
print("grader self-test passed")

**Goal.** Load the model and define `generate`, which answers a list of problems in batches and returns, for each, the text, the number of new tokens, whether the model finished (produced its end-of-turn token before the limit) and the loop flag. One line prints per batch with the running cut count, so a stuck run is visible within minutes.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL = "Qwen/Qwen3-0.6B"
tok = AutoTokenizer.from_pretrained(MODEL); tok.padding_side = "left"
DT = torch.float16 if dev == "cuda" else torch.float32
base = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DT).to(dev).eval()
EOS = set(np.atleast_1d(base.generation_config.eos_token_id).tolist())
SUFFIX = "\n\nPlease reason step by step, and put your final answer within \\boxed{}."

def gen_rows(model, texts, sampling):
    """New token ids for each prompt. If the GPU runs out of memory, the batch is split in half and retried."""
    x = tok(texts, return_tensors="pt", padding=True).to(dev); failed = False
    try:
        with torch.no_grad():
            out = model.generate(**x, max_new_tokens=MAX_NEW, pad_token_id=tok.pad_token_id, **sampling)
        return [out[r, x["input_ids"].shape[1]:].tolist() for r in range(len(texts))]
    except torch.cuda.OutOfMemoryError:
        failed = True
    # the retry sits outside the except block: inside it Python still holds the failed call's memory
    del x; gc.collect(); torch.cuda.empty_cache()
    if len(texts) == 1: raise RuntimeError("out of memory even at batch size 1")
    h = len(texts) // 2; print(f"  out of memory at batch size {len(texts)}: retrying as {h} + {len(texts) - h}", flush=True)
    return gen_rows(model, texts[:h], sampling) + gen_rows(model, texts[h:], sampling)

def generate(model, items, thinking, sampling, label=""):
    prompts = [tok.apply_chat_template([{"role": "user", "content": it["problem"] + SUFFIX}], tokenize=False, add_generation_prompt=True, enable_thinking=thinking) for it in items]
    order = sorted(range(len(items)), key=lambda i: len(prompts[i])); res = [None] * len(items); t0 = time.time(); cut = 0
    for b in range(0, len(order), BATCH):
        ii = order[b:b + BATCH]; rows = gen_rows(model, [prompts[i] for i in ii], sampling)
        for r, i in enumerate(ii):
            ids = rows[r]
            fin = any(t in EOS for t in ids)
            if fin: ids = ids[:min(j for j, t in enumerate(ids) if t in EOS)]
            text = tok.decode(ids, skip_special_tokens=True)
            res[i] = dict(text=text, n=len(ids), finished=fin, loop=looped(ids), correct=is_correct(text, items[i]["answer"], thinking))
            cut += (not fin)
        print(f"  {label:34s} batch {b // BATCH + 1}/{math.ceil(len(order) / BATCH)}  cut so far {cut}  [{time.time() - t0:5.0f} s]", flush=True)
    return res

def summarize(name, res):
    n = len(res); out = {}
    for key, f in (("accuracy", lambda r: r["correct"]), ("cut", lambda r: not r["finished"]), ("loop", lambda r: r["loop"])):
        k = sum(f(r) for r in res); lo, hi = wilson(k, n); out[key] = (k, n, lo, hi)
    print(f"{name:40s} accuracy {out['accuracy'][0]:3d}/{n} ({out['accuracy'][0]/n:.2f}, {out['accuracy'][2]:.2f} to {out['accuracy'][3]:.2f})   "
          f"cut {out['cut'][0]:3d}/{n} ({out['cut'][0]/n:.2f}, {out['cut'][2]:.2f} to {out['cut'][3]:.2f})   loop {out['loop'][0]:3d}/{n}")
    return out
def mcnemar(a, b):
    n10 = sum(x["correct"] and not y["correct"] for x, y in zip(a, b)); n01 = sum(y["correct"] and not x["correct"] for x, y in zip(a, b))
    return n10, n01, (binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0)

**Goal. Part 1.** Run the four arms of the unchanged model on the evaluation problems. The card's settings are the ones published with the model: thinking mode `temperature=0.6, top_p=0.95, top_k=20`; non-thinking mode `temperature=0.7, top_p=0.8, top_k=20`. The repetition-penalty arm is the cheapest decoding repair: greedy decoding with `repetition_penalty=1.1`, which lowers the score of tokens already seen.

In [ ]:
ARMS = {
    "thinking on, greedy":                (True,  dict(do_sample=False)),
    "thinking on, card settings":         (True,  dict(do_sample=True, temperature=0.6, top_p=0.95, top_k=20)),
    "thinking on, greedy + rep. penalty": (True,  dict(do_sample=False, repetition_penalty=1.1)),
    "thinking off, card settings":        (False, dict(do_sample=True, temperature=0.7, top_p=0.8, top_k=20)),
}
base_res, base_sum = {}, {}
torch.manual_seed(SEED)
for name, (thinking, sampling) in ARMS.items():
    base_res[name] = generate(base, eval_items, thinking, sampling, name)
    base_sum[name] = summarize(name, base_res[name])
json.dump({k: {m: list(v) for m, v in sm.items()} for k, sm in base_sum.items()}, open("19_loop_repair_part1.json", "w"), indent=1)   # kept even if Part 2 fails

**Goal.** Compare arms pairwise on the same problems with an exact McNemar test (it counts only the problems one arm solves and the other does not). The comparison that matters for the weight-editing story is the last: does the model with thinking on and sensible settings beat the model with thinking off? Read the **cut** column first: an arm with a high cut rate is mostly measuring the token limit.

In [ ]:
pairs = [("thinking on, card settings", "thinking on, greedy"), ("thinking on, greedy + rep. penalty", "thinking on, greedy"),
         ("thinking off, card settings", "thinking on, greedy"), ("thinking off, card settings", "thinking on, card settings")]
for a, b in pairs:
    n10, n01, p = mcnemar(base_res[a], base_res[b]); print(f"{a:36s} vs {b:36s}: only first right {n10:3d}, only second right {n01:3d}, exact McNemar p = {p:.2g}")

**Goal. Part 2.** Collect the LoRA's training data from the model itself. Run thinking mode with the card's settings on the GSM8K training problems and keep the traces that **finished and are correct** (rejection sampling). Training the model on its own successes teaches it what a finished reasoning trace looks like without any outside teacher; it is the ordinary-repair counterpart of the frontier-model edit.

In [ ]:
torch.manual_seed(SEED)
traces = generate(base, gsm_items, True, ARMS["thinking on, card settings"][1], "GSM8K traces")
kept = [(it, r) for it, r in zip(gsm_items, traces) if r["finished"] and r["correct"] and not r["loop"]]
print(f"kept {len(kept)} of {len(traces)} traces (finished, correct, no loop); mean length {np.mean([r['n'] for _, r in kept]) if kept else 0:.0f} tokens")
if SMOKE and not kept: kept = list(zip(gsm_items, traces))[:4]; print("SMOKE: using unfiltered traces so the training code runs")
assert kept, "no usable traces: raise N_TRACE or MAX_NEW"

**Goal.** Train the LoRA. Rank 8 on the attention projections, 2 epochs, learning rate 2e-4, batch of 8 sequences by gradient accumulation, loss on the model's answer only (not on the prompt). These settings were fixed before the first run. Afterward the adapter is merged into the weights, so the evaluated model is an ordinary checkpoint.

In [ ]:
from peft import LoraConfig, get_peft_model
train_model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(dev)
train_model.gradient_checkpointing_enable(); train_model.enable_input_require_grads()
lcfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM")
pm = get_peft_model(train_model, lcfg); pm.print_trainable_parameters()
def encode(it, r):
    p = tok.apply_chat_template([{"role": "user", "content": it["problem"] + SUFFIX}], tokenize=False, add_generation_prompt=True, enable_thinking=True)
    pi = tok(p, add_special_tokens=False)["input_ids"]; ci = tok(r["text"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = (pi + ci)[:1792]; lab = ([-100] * len(pi) + ci)[:1792]
    return torch.tensor(ids), torch.tensor(lab)
data = [encode(it, r) for it, r in kept]
EPOCHS, ACC, LR = (1 if SMOKE else 2), (2 if SMOKE else 8), 2e-4
opt = torch.optim.AdamW([p for p in pm.parameters() if p.requires_grad], lr=LR)
total = EPOCHS * math.ceil(len(data) / ACC); sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 1 - s / max(1, total)); step = 0
pm.train(); t0 = time.time()
for ep in range(EPOCHS):
    random.Random(SEED + ep).shuffle(data)
    for b in range(0, len(data), ACC):
        loss_sum = 0.0
        for ids, lab in data[b:b + ACC]:
            with torch.autocast(dev, dtype=torch.float16, enabled=(dev == "cuda")):
                loss = pm(input_ids=ids[None].to(dev), labels=lab[None].to(dev)).loss / len(data[b:b + ACC])
            loss.backward(); loss_sum += loss.item()
        torch.nn.utils.clip_grad_norm_(pm.parameters(), 1.0); opt.step(); sched.step(); opt.zero_grad(); step += 1
        print(f"  epoch {ep + 1} step {step}/{total} loss {loss_sum:.3f}  [{time.time() - t0:4.0f} s]", flush=True)
pm.save_pretrained("lora_adapter")   # keep the trained adapter before it is merged into the weights
merged = pm.merge_and_unload().to(DT).eval()
del pm, opt, sched, train_model, loss, data, base; gc.collect(); torch.cuda.empty_cache()   # free training memory before generating

**Goal.** Evaluate the LoRA-tuned model on the same evaluation problems under three of the arms: the loop-prone setting (thinking on, greedy), the card's thinking settings, and thinking off (which shows what the training cost where nothing was broken).

In [ ]:
tuned_res, tuned_sum = {}, {}
torch.manual_seed(SEED)
for name in ("thinking on, greedy", "thinking on, card settings", "thinking off, card settings"):
    thinking, sampling = ARMS[name]
    tuned_res[name] = generate(merged, eval_items, thinking, sampling, "LoRA " + name)
    tuned_sum[name] = summarize("LoRA: " + name, tuned_res[name])
print()
for name in tuned_res:
    n10, n01, p = mcnemar(tuned_res[name], base_res[name]); print(f"{name:30s}: LoRA only right {n10:3d}, base only right {n01:3d}, exact McNemar p = {p:.2g}")
json.dump({"base": {k: {m: list(v) for m, v in s.items()} for k, s in base_sum.items()}, "lora": {k: {m: list(v) for m, v in s.items()} for k, s in tuned_sum.items()},
           "settings": dict(model=MODEL, n_eval=N_EVAL, n_trace=N_TRACE, max_new=MAX_NEW, kept_traces=len(kept), seed=SEED, smoke=SMOKE)},
          open("19_loop_repair_results.json", "w"), indent=1)

**How to read the result.** Three questions, in this order. (1) In Part 1, how much of the thinking-on, greedy arm's cut rate disappears with the card's settings alone, and does the thinking-on arm then beat thinking off? If the card's settings already match the best arm, a repair beyond them has little left to fix on this model. (2) Does the repetition penalty remove loops, and what does it do to accuracy? (3) Does the LoRA lower the cut rate in the arms where the base model loops, and does it lose accuracy with thinking off? A repair that lowers the cut rate but also lowers accuracy has traded one failure for another.

**Limits.** One small model (Qwen3-0.6B, not the Qwen3.5-0.8B of the weight-editing case), one token limit, 100 problems unless you raise `N_EVAL`, so the Wilson intervals are wide: a gap smaller than about ten points between two arms is not distinguishable here. One LoRA configuration and one training seed. The traces come from GSM8K and the evaluation from MATH-500, which are different in difficulty. The grader is crude and the loop flag is a heuristic. None of this is evidence about the frontier-model edit itself; it measures only the ordinary repairs the edit was never compared with.

`19_loop_repair_results.json` is written next to the notebook. Download it (left sidebar, Files) into `data/results/` of the companion repo.

**Goal.** Save everything. The zip holds every generated answer (base and LoRA, with the problem, reference, correctness, cut and loop flags), all 300 GSM8K traces with the ones used for training marked, the trained LoRA adapter, the two result tables, and `run_manifest.json` with the settings, the software and GPU, and the revisions of the model and datasets. On Kaggle the files are collected from the working directory automatically; in Colab the zip downloads, and the files also stay in the Files panel until the runtime is recycled.

In [ ]:
import sys, platform, hashlib, zipfile, shutil, datetime, importlib.metadata as md
COPY_TO_DRIVE = False   # True: also copy the zip to Google Drive (Colab asks you to sign in)

def package_version(name):
    try: return md.version(name)
    except Exception: return None

def hub_revision(kind, repo):
    """The commit of a Hugging Face model or dataset that was downloaded, so the run can be reproduced."""
    try:
        from huggingface_hub import HfApi
        api = HfApi(); return (api.dataset_info(repo) if kind == "dataset" else api.model_info(repo)).sha
    except Exception as e:
        return "unavailable: " + repr(e)[:60]

def sha256_of(obj): return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()

def environment():
    gpu = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
    return dict(finished=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"), python=sys.version.split()[0], platform=platform.platform(),
                torch=torch.__version__, cuda=torch.version.cuda, gpu=gpu.name if gpu else None, gpu_gb=round(gpu.total_memory / 1e9, 1) if gpu else None,
                packages={p: package_version(p) for p in ("transformers", "peft", "accelerate", "bitsandbytes", "huggingface_hub", "scipy", "numpy", "pandas")}, smoke=SMOKE)

def save_bundle(zip_name, paths, manifest):
    """Write run_manifest.json, zip it with the listed files and folders, print what is inside, offer the download."""
    json.dump(manifest, open("run_manifest.json", "w"), indent=1, default=str)
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
        for p in ["run_manifest.json"] + paths:
            if os.path.isdir(p):
                for root, _, names in os.walk(p):
                    for n in names: z.write(os.path.join(root, n))
            elif os.path.exists(p): z.write(p)
            else: print("missing, not zipped:", p)
    with zipfile.ZipFile(zip_name) as z:
        print(f"wrote {zip_name} ({os.path.getsize(zip_name) / 1e6:.1f} MB). Contents:")
        for i in z.infolist(): print(f"  {i.file_size / 1e6:8.2f} MB  {i.filename}")
    if COPY_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive"); shutil.copy(zip_name, "/content/drive/MyDrive/"); print("copied to My Drive")
    try:
        from google.colab import files as colab_files; colab_files.download(zip_name)
    except Exception:
        print("Not in Colab, or the browser blocked the download: take the zip from the Files panel.")

def rows(items, res): return [{**it, **r} for it, r in zip(items, res)]
json.dump({k: rows(eval_items, v) for k, v in base_res.items()}, open("19_generations_base.json", "w"))
json.dump({k: rows(eval_items, v) for k, v in tuned_res.items()}, open("19_generations_lora.json", "w"))
json.dump(rows(gsm_items, traces), open("19_gsm8k_traces.json", "w"))
kept_idx = [i for i, r in enumerate(traces) if r["finished"] and r["correct"] and not r["loop"]]
manifest = dict(environment=environment(), model=MODEL, model_revision=hub_revision("model", MODEL),
                settings=dict(n_eval=N_EVAL, n_trace=N_TRACE, max_new=MAX_NEW, batch=BATCH, seed=SEED, lora=dict(r=8, alpha=16, dropout=0.05, epochs=EPOCHS, lr=LR, accumulation=ACC)),
                math500_revision=hub_revision("dataset", "HuggingFaceH4/MATH-500"), gsm8k_revision=hub_revision("dataset", "openai/gsm8k"),
                eval_items_sha256=sha256_of(eval_items), trace_indices_used_for_training=kept_idx, traces_total=len(traces))
save_bundle("19_loop_repair_bundle.zip", ["19_loop_repair_results.json", "19_loop_repair_part1.json", "19_generations_base.json", "19_generations_lora.json", "19_gsm8k_traces.json", "lora_adapter"], manifest)